In [1]:
import torch

if torch.cuda.is_available():
    print(f"CUDA is available. You have {torch.cuda.device_count()} GPU(s).")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. You are using the CPU.")

CUDA is available. You have 1 GPU(s).
GPU Name: NVIDIA RTX 1000 Ada Generation Laptop GPU


In [2]:
import napari
from qtpy.QtWidgets import QWidget, QVBoxLayout, QHBoxLayout, QPushButton, QLabel, QStackedWidget, QMessageBox, QLineEdit, QComboBox
from magicgui.widgets import ComboBox, FileEdit
import os
import glob
import numpy as np
import tifffile


In [ ]:
class MyTool(QWidget):
    def __init__(self, viewer):
        super().__init__()
        self.viewer = viewer
        self.state = {}
        self.current_image = None

        # Holds all pages
        self.pages = QStackedWidget()
        self.upload_page = self.create_upload_page()
        self.pages.addWidget(self.upload_page)

        layout = QVBoxLayout()
        layout.addWidget(self.pages)
        self.setLayout(layout)

    # PAGE 1 - Upload Data: Either Image or Stack
    # self.upload_type = "Image" or "Folder"; self.inputs maps name -> input widgets
    # (self.inputs[name]["image"] / ["folder"]). self.label_status / self.tracks_status
    # flag whether inputs are pre-labelled / pre-tracked.
    def change_upload_type(self):
        is_image = self.upload_type.value == "Image"
        
        for widgets in self.inputs.values():
            widgets["image"].visible = is_image
            widgets["folder"].visible = not is_image
            if is_image:
                widgets["label"].setText(widgets["image_label"])
            else:
                widgets["label"].setText(widgets["folder_label"])


    def create_upload_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.addWidget(QLabel("Upload Your Data"))

        # Image / Folder selector
        self.upload_type = ComboBox(label="Input type:", choices=["Image", "Folder"], value="Image")
        layout.addWidget(self.upload_type.native)

        # Store our input widgets
        self.inputs = {}

        # Add the three rows
        self.add_upload_row(layout, "original", "Original Image (Stack)", "(Folder of) Original Images")
        self.add_upload_row(layout, "labels", "(Stack of) Labels", "(Folder of) Labels")
        self.add_upload_row(layout, "tracks", "(Stack of) Linked Tracks", "(Folder of) Linked Tracks")

        # Update everything when Image/Folder changes
        self.upload_type.changed.connect(self.change_upload_type)

        # Next button
        next_button = QPushButton("Next →")
        next_button.clicked.connect(self.check_upload_page)
        layout.addWidget(next_button)

        return page

    def add_upload_row(self, layout, name, image_label_text, folder_label_text):
        row = QHBoxLayout()
        label = QLabel(image_label_text)
        image_input = FileEdit(label="", mode="r", filter="TIFF (*.tif *.tiff)")
        folder_input = FileEdit(label="", mode="d")

        # Default to image input, so hide the folder input intiailly 
        folder_input.visible = False

        row.addWidget(label)
        row.addWidget(image_input.native)
        row.addWidget(folder_input.native)

        layout.addLayout(row)

        self.inputs[name] = {
            "label": label,
            "image": image_input,
            "folder": folder_input,
            "image_label": image_label_text,
            "folder_label": folder_label_text
        }
        
    def is_empty(self, value):
        # magicgui FileEdit returns a Path; an unset field is Path('.'), not "".
        return str(value).strip() in ("", ".")

    def tyx_size(self, tif_path):
        # (T, Y, X) sizes only; channel axis ignored so (T,C,Y,X) matches (T,1,Y,X).
        with tifffile.TiffFile(tif_path) as tif:
            series = tif.series[0]
            if "S" in series.axes:
                t_dim = series.axes.index("S")
            elif "T" in series.axes:
                t_dim = series.axes.index("T")
            else:
                t_dim = 1
            sizes = dict(zip(series.axes, series.shape))
        return (t_dim, sizes.get("Y"), sizes.get("X"))

    def to_tcyx(self, tif_path):
        # Return array as (T, C, Y, X); missing T/C axes are inserted as size 1.
        with tifffile.TiffFile(tif_path) as tif:
            series = tif.series[0]
            image_array = series.asarray()
            raw_axes = series.axes
            dims = list(raw_axes)
        # tifffile may label a stray frame/sample axis 'S'/'I'/'Q'; treat it as time.
        dims = ["T" if a in ("S", "I", "Q") else a for a in dims]
        # Drop any remaining non-standard singleton axes (e.g. a size-1 Z).
        for axis, size in list(zip(list(dims), image_array.shape)):
            if axis not in ("T", "C", "Y", "X") and size == 1:
                index = dims.index(axis)
                image_array = np.squeeze(image_array, axis=index)
                dims.pop(index)
        if "Y" not in dims or "X" not in dims or set(dims) - {"T", "C", "Y", "X"}:
            raise ValueError(f"Unsupported TIFF axes '{raw_axes}' for {tif_path}.")
        for axis in ("C", "T"):
            if axis not in dims:
                image_array = np.expand_dims(image_array, 0)
                dims.insert(0, axis)
        order = [dims.index(a) for a in ("T", "C", "Y", "X")]
        return np.transpose(image_array, order)

    def check_upload_page(self):
        try:
            ###################TO DO ------ CAN'T JUST HAVE A SINGLE CHANNEL IMAGE ############
            ############ IF T && Z -> FAIL
            ### FAILURE MODES - no inputs
            if (self.upload_type.value == "Image" and self.is_empty(self.inputs["original"]["image"].value)) or (self.upload_type.value == "Folder" and self.is_empty(self.inputs["original"]["folder"].value)):
                QMessageBox.warning(self, "Input Error", "Please select an original image/folder.")
                return
            ########## FOLDER FAILURE MODES: empty folders, mismatched length in folders, same folder listed multiple times
            if self.upload_type.value == "Folder":
                folders = {
                    "Original": self.inputs["original"]["folder"].value,
                    "Labels": self.inputs["labels"]["folder"].value,
                    "Tracks": self.inputs["tracks"]["folder"].value,
                }
                # Unset FileEdit is Path('.'); normalise blanks to "" so they aren't treated as a real (duplicate) folder.
                folders = {name: ("" if self.is_empty(value) else str(value)) for name, value in folders.items()}
                # Check for duplicate folders
                folder_paths = [folder for folder in folders.values() if folder]
                if len(folder_paths) != len(set(folder_paths)):
                    QMessageBox.warning(self, "Input Error", "The same folder has been listed multiple times.")
                    return

                n_original = len(glob.glob(os.path.join(folders["Original"], "*.tif"))) + len(glob.glob(os.path.join(folders["Original"], "*.tiff")))
                n_labels = len(glob.glob(os.path.join(folders["Labels"], "*.tif"))) + len(glob.glob(os.path.join(folders["Labels"], "*.tiff"))) if folders["Labels"] else 0
                n_tracks = len(glob.glob(os.path.join(folders["Tracks"], "*.tif"))) + len(glob.glob(os.path.join(folders["Tracks"], "*.tiff"))) if folders["Tracks"] else 0
                folder_lengths = [n_original]
                if n_labels:
                    folder_lengths.append(n_labels)
                if n_tracks:
                    folder_lengths.append(n_tracks)
                if set(folder_lengths) == {0}:
                    QMessageBox.warning(self, "Input Error", "All selected folders are empty.")
                    return
                if len(set(folder_lengths)) > 1:
                    QMessageBox.warning(self, "Input Error", "Folders contain different numbers of images.")
                    return

                self.original_images = glob.glob(os.path.join(folders["Original"], "*.tif")) + glob.glob(os.path.join(folders["Original"], "*.tiff"))
                self.label_images = glob.glob(os.path.join(folders["Labels"], "*.tif")) + glob.glob(os.path.join(folders["Labels"], "*.tiff")) if folders["Labels"] else []
                self.track_images = glob.glob(os.path.join(folders["Tracks"], "*.tif")) + glob.glob(os.path.join(folders["Tracks"], "*.tiff")) if folders["Tracks"] else []
            else:
                ######### IMAGE FAILURE MODES - same path repeated
                self.original_images = [] if self.is_empty(self.inputs["original"]["image"].value) else [self.inputs["original"]["image"].value]
                self.label_images = [] if self.is_empty(self.inputs["labels"]["image"].value) else [self.inputs["labels"]["image"].value]
                self.track_images = [] if self.is_empty(self.inputs["tracks"]["image"].value) else [self.inputs["tracks"]["image"].value]

                paths = [str(p) for p in (self.inputs["original"]["image"].value, self.inputs["labels"]["image"].value, self.inputs["tracks"]["image"].value) if not self.is_empty(p)]
                if len(paths) != len(set(paths)):
                    QMessageBox.warning(self, "Input Error", "You've listed the same file in multiple categories! Ensure paths are unique.")
                    return

            if (self.upload_type.value == "Image" and self.is_empty(self.inputs["labels"]["image"].value)) or (self.upload_type.value == "Folder" and self.is_empty(self.inputs["labels"]["folder"].value)):
                self.label_status = False
            else:
                self.label_status = True
            if (self.upload_type.value == "Image" and self.is_empty(self.inputs["tracks"]["image"].value)) or (self.upload_type.value == "Folder" and self.is_empty(self.inputs["tracks"]["folder"].value)):
                self.tracks_status = False
            else:
                self.tracks_status = True

            self.original_images.sort()
            self.label_images.sort()
            self.track_images.sort()
            for i, image in enumerate(self.original_images):
                original_shape = self.tyx_size(image)
                label_shape = self.tyx_size(self.label_images[i]) if self.label_status else None
                track_shape = self.tyx_size(self.track_images[i]) if self.tracks_status else None
                shapes = [original_shape]
                if label_shape is not None:
                    shapes.append(label_shape)
                if track_shape is not None:
                    shapes.append(track_shape)
                if len(set(shapes)) > 1:
                    QMessageBox.warning(self, "Input Error", f"(T, Y, X) sizes do not match for {image}:\n"
                        f"Original: {original_shape}\n"
                        f"Label: {label_shape if label_shape is not None else 'N/A'}\n"
                        f"Track: {track_shape if track_shape is not None else 'N/A'}"
                    )
                    return

            if getattr(self, "threshold_page", None) is not None:
                self.pages.removeWidget(self.threshold_page)
                self.threshold_page.deleteLater()
            self.threshold_page = self.create_threshold_page()
            self.pages.addWidget(self.threshold_page)
            self.pages.setCurrentIndex(self.pages.indexOf(self.threshold_page))
        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")


    ####### PAGE 2 - Threshold and Channel Assignment
    # --------------------------
    def map_channels_to_plotting_colours(self, channel_name):
        channel_name = channel_name.lower()
        colour_map = {
            "dodgerblue": ["dapi", "hoechst", "405", "blue"],
            "limegreen":  ["fitc", "gfp", "488", "green"],
            "red":        ["tritc", "rfp", "561", "555", "mcherry", "mscarlet", "tomato", "red"],
            "deeppink":   ["cy5", "647", "640", "far red", "far-red", "magenta"],
            "darkorange": ["cy3", "orange", "561", "568", "yellow", "yfp"],
        }
        for colour, keywords in colour_map.items():
            if any(keyword in channel_name for keyword in keywords):
                return colour
        return "gray"
    
    def check_and_apply_thresholds(self):
        print("tbc")

    def add_threshold_row(self, layout,  default_channel=0, default_name="", default_threshold="Otsu"):
        row = QHBoxLayout()
        n_channels = self.current_image.shape[1]
        # Channel dropdown: 0 ... n_channels-1
        channel_input = QComboBox()
        channel_input.addItems([str(i) for i in range(n_channels)])
        channel_input.setCurrentIndex(default_channel)

        # Channel name
        name_input = QLineEdit()
        name_input.setText(default_name)
        name_input.setPlaceholderText("Channel name")

        # Threshold method
        threshold_input = QComboBox()
        threshold_input.addItems(["Mean", "Otsu", "Yen", "Triangle", "Minimum"])
        threshold_input.setCurrentText(default_threshold)

        row.addWidget(channel_input)
        row.addWidget(name_input)
        row.addWidget(threshold_input)

        layout.addLayout(row)

    def create_threshold_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)

        layout.addWidget(QLabel("Channels and Thresholds"))


        try:
            self.current_image = self.to_tcyx(self.original_images[0])
            n_channels = self.current_image.shape[1]

            # Brightfield / segmentation channel (# + name): only needed when the user hasn't supplied labels.
            if not self.label_status:
                layout.addWidget(QLabel("Channel To Segment Cells/Nuclei From"))
            else:
                layout.addWidget(QLabel("Channel Your Labels were Built From"))
            bf_row = QHBoxLayout()
            bf_channel_input = QComboBox()
            bf_channel_input.addItems([str(i) for i in range(n_channels)])
            bf_channel_input.setCurrentIndex(0)
            bf_name_input = QLineEdit()
            bf_name_input.setPlaceholderText("Brightfield")
            bf_row.addWidget(bf_channel_input)
            bf_row.addWidget(bf_name_input)
            layout.addLayout(bf_row)

            layout.addWidget(QLabel("Fluorescent Channels to Analyse"))

            for channel_index in range(n_channels - 1):
                self.add_threshold_row(layout, default_channel=channel_index + 1, default_name=f"Channel {channel_index + 1}")


            ### BUTTON FOR APPLYING THRESHOLDS
            apply_thresholds_button = QPushButton("Apply Thresholds")
            apply_thresholds_button.clicked.connect(self.check_and_apply_thresholds)
            layout.addWidget(apply_thresholds_button)
            ##### FAILURE MODES: Multiple inputs with the same channel index
            
            #### If success: add thresholded blobs
            
            
            self.viewer.layers.clear()
            current_image = self.to_tcyx(self.original_images[0])
            self.viewer.add_image(current_image, channel_axis=1)
            
            
            
            
            
            
            
            
            
            
            
            
        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")

        back = QPushButton("← Back")
        back.clicked.connect(lambda: self.pages.setCurrentIndex(0))
        layout.addWidget(back)

        return page

    # --------------------------
    # PAGE 3
    # --------------------------

    def create_cellpose_page(self):

        page = QWidget()
        layout = QVBoxLayout(page)

        try:
            layout.addWidget(
                QLabel("Ready to run!")
            )

            back = QPushButton("← Back")
            run = QPushButton("Run analysis")

            back.clicked.connect(
                lambda: self.pages.setCurrentIndex(1)
            )

            run.clicked.connect(self.run_analysis)

            layout.addWidget(back)
            layout.addWidget(run)
        except Exception as exception:
            import traceback
            traceback.print_exc()
            QMessageBox.warning(self, "Error", f"{type(exception).__name__}: {exception}")

        return page


viewer = napari.Viewer()

tool = MyTool(viewer)

viewer.window.add_dock_widget(
    tool,
    name="FluoroFate",
    area="right",
)

napari.run()